# Week 6 Exercise: Simple Fine-Tuning Demo

This notebook demonstrates **fine-tuning a model** with a minimal, end-to-end workflow:

1. **Data**: Use the product pricer dataset (`human_in.csv` and `human_out.csv` in this folder) — product descriptions as input, prices as target.
2. **Preparation**: Build prompt/completion pairs, split into train/validation, and convert to OpenAI chat JSONL format.
3. **Fine-tuning**: Optionally run a real OpenAI fine-tuning job or a **simulation** (no API cost) to show the pipeline.
4. **Performance**: Display metrics (e.g. Mean Absolute Error) and a simple plot comparing predicted vs actual prices.

## 1. Imports and setup

In [8]:
import os
import re
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
%matplotlib inline

## 2. Load data from CSVs

We use `human_in.csv` (product descriptions) and `human_out.csv` (same descriptions with human-provided prices) in this folder. Rows are aligned: row i in input matches row i in output.

In [9]:
# Paths: notebook and data are in week6 folder
DATA_DIR = Path(".")
PATH_IN = DATA_DIR / "human_in.csv"
PATH_OUT = DATA_DIR / "human_out.csv"

if not PATH_IN.exists():
    # If run from repo root
    DATA_DIR = Path("week6")
    PATH_IN, PATH_OUT = DATA_DIR / "human_in.csv", DATA_DIR / "human_out.csv"

print("human_in.csv exists:", PATH_IN.exists())
print("human_out.csv exists:", PATH_OUT.exists())

human_in.csv exists: True
human_out.csv exists: True


In [10]:
# Load CSVs (multiline quoted fields supported)
df_in = pd.read_csv(PATH_IN, header=None, engine="python")
df_out = pd.read_csv(PATH_OUT, header=None, engine="python")

# Column 0 = product text, column 1 = 0 (input) or price (output)
prompts = df_in[0].astype(str).str.strip()
prices = pd.to_numeric(df_out[1], errors="coerce")

df = pd.DataFrame({"prompt": prompts, "completion": prices})
df = df.dropna()
df = df[df["prompt"].str.len() > 0]
df = df.reset_index(drop=True)

print(f"Loaded {len(df)} prompt–price pairs")
print(df.head(2))

Loaded 100 prompt–price pairs
                                              prompt  completion
0  Title: Excess V2 Distortion/Modulation Pedal  ...         120
1  Title: Telpo Headlight Assembly for 2015‑2017 ...         300


## 3. Train/validation split and JSONL for fine-tuning

Split the data and convert to OpenAI chat format: system message, user message (product description), assistant message (price only).

In [11]:
RANDOM_SEED = 42
train_df, val_df = train_test_split(df, test_size=0.15, random_state=RANDOM_SEED)

SYSTEM_MSG = "You are a product pricing assistant. Reply with the price in dollars only, no explanation."
ASSISTANT_PREFIX = "Price is $"

def build_messages(row):
    return [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": f"{ASSISTANT_PREFIX}{row['completion']:.0f}"},
    ]

def to_jsonl(df, path):
    with open(path, "w") as f:
        for _, row in df.iterrows():
            rec = {"messages": build_messages(row)}
            f.write(json.dumps(rec) + "\n")

to_jsonl(train_df, "fine_tune_train.jsonl")
to_jsonl(val_df, "fine_tune_val.jsonl")
print(f"Train: {len(train_df)} examples, Val: {len(val_df)} examples")
print("Sample train message:", build_messages(train_df.iloc[0]))

Train: 85 examples, Val: 15 examples
Sample train message: [{'role': 'system', 'content': 'You are a product pricing assistant. Reply with the price in dollars only, no explanation.'}, {'role': 'user', 'content': 'Title: Raven Pro Document Scanner Stand  \nCategory: Accessories  \nBrand: Raven  \nDescription: Elevates the Raven Pro Document Scanner to provide a comfortable seated viewing angle.  \nDetails: Enhances visibility of the LCD touchscreen for easy use and includes a stable, compact design.'}, {'role': 'assistant', 'content': 'Price is $12'}]


## 4. Fine-tuning (simulation or real)

We use **GPT-4.1 nano** (`gpt-4.1-nano-2025-04-14`) — OpenAI's smallest, cheapest model for fine-tuning (~$1.50 per 1M training tokens), so runs are low-cost and suitable for a laptop workflow.

Set `SIMULATE = True` to run without API calls (no cost). Set `SIMULATE = False` and ensure `OPENAI_API_KEY` is set to run a real fine-tuning job.

In [ ]:
SIMULATE = False  # Set to False to run real fine-tuning (uses API credits)
FT_MODEL_ID = None  # Will be set after job completes (simulated or real)

if SIMULATE:
    import time
    print("Simulation mode: skipping API calls.")
    for step in ["Validating files", "Training epoch 1/2", "Training epoch 2/2"]:
        print(f"  {step} ...")
        time.sleep(0.5)
    FT_MODEL_ID = "ft:gpt-4.1-nano:week6-demo-simulated"
    print("Done. Using simulated model id for performance section.")
else:
    import time
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    with open("fine_tune_train.jsonl", "rb") as f:
        train_file = client.files.create(file=f, purpose="fine-tune")
    with open("fine_tune_val.jsonl", "rb") as f:
        val_file = client.files.create(file=f, purpose="fine-tune")
    # Fine-tuning: use a versioned base model. GPT-4.1 nano is smallest/cheapest (~$1.50/1M training tokens).
    job = client.fine_tuning.jobs.create(
        model="gpt-4.1-nano-2025-04-14",
        training_file=train_file.id,
        validation_file=val_file.id,
        hyperparameters={"n_epochs": 2, "learning_rate_multiplier": 1.0},
        suffix="week6-demo",
    )
    print("Job id:", job.id)
    while job.status in ("validating_files", "queued", "running"):
        time.sleep(30)
        job = client.fine_tuning.jobs.retrieve(job.id)
        print("  Status:", job.status)
    if job.status == "succeeded":
        FT_MODEL_ID = job.fine_tuned_model
        print("Fine-tuned model:", FT_MODEL_ID)
    else:
        print("Job did not succeed. Status:", job.status)

Job id: ftjob-OJg2jLW6LoXNF1fmBbzpBI98
  Status: validating_files
  Status: validating_files


## 5. Performance of the fine-tuned model

We evaluate on the validation set:
- **If simulating**: we mock predictions (actual + small noise) to show the evaluation pipeline and metrics.
- **If you ran a real job**: the previous cell will set `FT_MODEL_ID`; re-run from here to call the API and compute real MAE and plot.

In [ ]:
eval_size = min(50, len(val_df))
val_sub = val_df.sample(n=eval_size, random_state=RANDOM_SEED)

def extract_price(text):
    if isinstance(text, (int, float)):
        return float(text)
    s = str(text).replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else 0.0

actuals = val_sub["completion"].tolist()

In [ ]:
# Use real API only when we have a non-simulated fine-tuned model id
use_api = FT_MODEL_ID and "simulated" not in str(FT_MODEL_ID)

if use_api:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    predictions = []
    for _, row in val_sub.iterrows():
        r = client.chat.completions.create(
            model=FT_MODEL_ID,
            messages=[
                {"role": "system", "content": SYSTEM_MSG},
                {"role": "user", "content": row["prompt"]},
            ],
            max_tokens=16,
        )
        predictions.append(extract_price(r.choices[0].message.content))
else:
    # Simulated: small noise around actual to show evaluation pipeline
    predictions = [p + np.random.uniform(-8, 8) for p in actuals]
    predictions = [max(0, p) for p in predictions]

mae = mean_absolute_error(actuals, predictions)
print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Evaluated on {eval_size} validation examples.")

In [ ]:
# Plot: predicted vs actual
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(actuals, predictions, alpha=0.7, label="Predictions")
max_val = max(max(actuals), max(predictions))
ax.plot([0, max_val], [0, max_val], "k--", label="Perfect (y=x)")
ax.set_xlabel("Actual price ($)")
ax.set_ylabel("Predicted price ($)")
ax.set_title(f"Fine-tuned model performance (MAE = ${mae:.2f})")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## Summary

- **Data**: Product descriptions and prices from `human_in.csv` / `human_out.csv` in this folder.
- **Pipeline**: Train/val split → JSONL (chat format) → optional real or simulated fine-tuning.
- **Performance**: Validation MAE and predicted-vs-actual plot show how well the (simulated or real) fine-tuned model performs. For a real run, set `SIMULATE = False`, run the job; when it completes, re-run the performance cells to get real MAE and plot.